## Execute Semantic Similarity Queries in Azure Cosmos DB

### Installing Libraries and Utilities

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv openai==2.38.0

### Setting up your Environment

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

# fetching the cosmosdb configuration from environment variables
cosmosdb_endpoint = os.getenv("COSMOSDB_ENDPOINT")
cosmosdb_key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME") + "Vector"

# fetching the azure openai configuration from environment variables
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Creating the Cosmos DB Client

In [ ]:
from azure.cosmos import CosmosClient
from azure.cosmos import PartitionKey

client = CosmosClient(cosmosdb_endpoint, cosmosdb_key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Creating the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(client, text):
    
    response = client.embeddings.create(
        input=text,
        model = embedding_model_name
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

### Execute Your First Vector Search Query

In [ ]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.content,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}, similarity score: {}".format(item['name'], item['category'], item['SimilarityScore']))
    print("-----------")
    print("content: {}".format(item['content']))
    print("====================================")
    print("\n\n")

### Apply a Similarity Threshold to Filter Results

In [ ]:
query = """
SELECT TOP 5
    c.name,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
WHERE VectorDistance(
    c.vector,
    @queryVector
) > 0.7
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print(
        item["name"],
        item["SimilarityScore"]
    )

### Implement a Brute-Force Similarity Search

In [ ]:
query = """
SELECT TOP 5
    c.name,
    VectorDistance(
        c.vector,
        @queryVector,
        true
    ) AS SimilarityScore
FROM c
ORDER BY VectorDistance(
    c.vector,
    @queryVector,
    true
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print(
        item["name"],
        item["SimilarityScore"]
    )

### Perform Vector Search with Metadata Filtering

In [ ]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.dietaryTags,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
WHERE ARRAY_CONTAINS(
    c.dietaryTags,
    "Vegetarian"
)
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}, dietary tags: {}, similarity score: {}".format(
        item['name'],
        item['category'],
        item['dietaryTags'],
        item['SimilarityScore']
    ))

### Optimize Search with Partitioning 

In [ ]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.dietaryTags,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
WHERE ARRAY_CONTAINS(
    c.dietaryTags,
    "Vegetarian"
)
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "high protein food")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    partition_key="Smoothies",
)

for item in results:
    print("name: {}, category: {}, dietary tags: {}, similarity score: {}".format(
        item['name'],
        item['category'],
        item['dietaryTags'],
        item['SimilarityScore']
    ))